# 📊 Data Preparation and Exploratory Data Analysis

**Project:** Retail Sales Forecasting  
**Author:** [Your Name]  
**Date:** October 2025

## 🎯 Objectives
1. Load and examine the retail sales data
2. Handle missing values and data quality issues
3. Perform exploratory data analysis (EDA)
4. Engineer features for time series forecasting
5. Prepare data for modeling

---
## 📥 Step 1: Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

---
## 📂 Step 2: Load the Data

**Note:** Upload your `train.csv` file using the file upload button 📁 in Google Colab before running this cell.

In [ ]:
# Load the training data
df = pd.read_csv('train.csv')

In [ ]:
# Display basic information
print("📋 Dataset Information:\n")
df.info()

---
## 🔍 Step 3: Data Quality Assessment

In [ ]:
# Check for missing values
print("🔍 Missing Values Analysis:\n")
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

In [ ]:
# Statistical summary
print("📈 Statistical Summary:\n")
df.describe()

In [ ]:
# Check date range
df['date'] = pd.to_datetime(df['date'])

print("📅 Date Range Analysis:\n")
print(f"Start date: {df['date'].min()}")
print(f"End date: {df['date'].max()}")
print(f"Total days: {(df['date'].max() - df['date'].min()).days} days")
print(f"Unique dates: {df['date'].nunique()}")

# Check for missing dates
date_range = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='D')
missing_dates = set(date_range) - set(df['date'].unique())
print(f"\nMissing dates: {len(missing_dates)}")
if len(missing_dates) > 0:
    print(f"⚠️ Warning: {len(missing_dates)} dates are missing from the dataset")

---
## 🧹 Step 4: Data Cleaning

**Decisions made:**
1. Convert date column to datetime format ✅
2. Handle any missing unit_sales values
3. Remove or handle outliers (if necessary)
4. Ensure data types are correct

In [ ]:
# Handle missing values in unit_sales (our target variable)
if df['unit_sales'].isnull().sum() > 0:
    print(f"⚠️ Found {df['unit_sales'].isnull().sum()} missing values in unit_sales")
    print("🔧 Filling with 0 (assuming no sales on those days)")
    df['unit_sales'] = df['unit_sales'].fillna(0)
else:
    print("✅ No missing values in unit_sales")

---
## 📊 Step 5: Aggregate Data for Time Series

We'll aggregate sales by date to create a daily time series for forecasting.

In [ ]:
# Aggregate sales by date
daily_sales = df.groupby('date').agg({
    'unit_sales': 'sum',
    'onpromotion': 'sum',
    'store_nbr': 'count'  # Number of transactions
}).reset_index()

---
## 📈 Step 6: Exploratory Data Analysis (EDA)

### 6.1 Time Series Visualization

In [ ]:
# Plot full time series
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Plot 1: Full time series
axes[0].plot(daily_sales['date'], daily_sales['total_sales'], color='steelblue', linewidth=1, alpha=0.8)
axes[0].set_title('📊 Total Daily Sales - Full Time Series', fontsize=16, fontweight='bold', pad=20)
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Total Daily Sales (units)', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

### 6.2 Seasonal Patterns Analysis

In [ ]:
# Add time-based features for analysis
daily_sales['year'] = daily_sales['date'].dt.year
daily_sales['month'] = daily_sales['date'].dt.month
daily_sales['day_of_week'] = daily_sales['date'].dt.dayofweek
daily_sales['day_name'] = daily_sales['date'].dt.day_name()
daily_sales['month_name'] = daily_sales['date'].dt.month_name()

### 6.3 Trend Analysis

In [ ]:
# Calculate rolling averages to identify trends
daily_sales['sales_7d_ma'] = daily_sales['total_sales'].rolling(window=7, center=True).mean()
daily_sales['sales_30d_ma'] = daily_sales['total_sales'].rolling(window=30, center=True).mean()
daily_sales['sales_90d_ma'] = daily_sales['total_sales'].rolling(window=90, center=True).mean()

---
## 🔧 Step 7: Feature Engineering

Create additional features that might help improve forecast accuracy.

In [ ]:
# Lag features (previous sales)
daily_sales['lag_1'] = daily_sales['total_sales'].shift(1)  # Previous day
daily_sales['lag_7'] = daily_sales['total_sales'].shift(7)  # Same day last week
daily_sales['lag_14'] = daily_sales['total_sales'].shift(14)  # Same day 2 weeks ago

---
## 💾 Step 8: Save Prepared Data

In [ ]:
# Save the prepared dataset
daily_sales.to_csv('daily_sales_prepared.csv', index=False)